# Improved Sequence Pattern Extraction for Smart Home Routine Discovery

## Overview

Smart homes generate continuous streams of events from sensors and connected
devices.

Examples include

- Presence sensors
- Smart lights
- Smart fans
- Door locks
- Window sensors
- Smart plugs

While individual events provide information about the current state of the
household, they rarely reveal behavioural routines on their own.

Instead, routines emerge as **ordered sequences of events** occurring close
together in time.

Examples include

Morning Departure

Bedroom Presence → LEAVE

↓

Bedroom Fan → OFF

↓

Bedroom Light → OFF

↓

Main Door → LOCK

Night Routine

Living Room TV → OFF

↓

Bedroom Light → ON

↓

Bedroom Presence → ENTER

↓

Bedroom AC → ON

Discovering these recurring sequences enables the smart home to learn user
behaviour automatically.

The objective of this notebook is to build a production-ready deterministic
sequence mining engine capable of extracting such routines from historical
event logs.

# Motivation

The original sequence extraction algorithm relied on exact matching.

For example,

Routine A

LEAVE

↓

FAN OFF

↓

LIGHT OFF

↓

LOCK

Routine B

LEAVE

↓

LIGHT OFF

↓

FAN OFF

↓

LOCK

Although these represent the same household behaviour,
the original algorithm would classify them as different patterns.

Similarly,

LEAVE

↓

FAN OFF

↓

LIGHT OFF

and

LEAVE

↓

LIGHT OFF

would also be treated as completely different routines.

Real households rarely execute routines identically every day.

Sensor delays,
optional actions,
duplicate events,
and user behaviour naturally introduce small variations.

Therefore, the objective is to discover **behaviourally similar**
rather than **identical** routines.

# Design Goals

The improved algorithm is designed according to the following principles.

1. Household Independent

The algorithm should work for any smart home regardless of the devices
installed.

No appliance-specific rules should be required.

---

2. Deterministic

The same event history should always produce the same patterns.

---

3. Explainable

Every discovered routine should be traceable back to the historical events
that produced it.

---

4. Scalable

The algorithm should comfortably process event histories spanning
30 days containing thousands of events.

---

5. Noise Tolerant

Minor variations such as

- reordered actions
- optional actions
- duplicate sensor updates

should not create completely new routines.

---

6. Computationally Efficient

The algorithm should avoid expensive all-to-all comparisons whenever possible
and instead compare only likely candidate sessions.

# Overall Pipeline

The complete sequence mining pipeline consists of thirteen stages.

Historical Event Logs
        │
        ▼
Chronological Sorting
        │
        ▼
Adaptive Sessionization
        │
        ▼
Duplicate Event Compression
        │
        ▼
Canonical Session Representation
        │
        ▼
Candidate Index Generation
        │
        ▼
Pairwise Session Similarity
        │
        ├── Sequence Similarity
        ├── Temporal Similarity
        └── Length Similarity
        ▼
Adaptive Neighbour Selection
        │
        ▼
Similarity Graph
        │
        ▼
Connected Components
        │
        ▼
Representative Routine
        │
        ▼
Pattern Statistics
        │
        ├── Support
        ├── Circular Mean Start Time
        ├── Circular Standard Deviation
        ├── Cluster Stability
        └── Completeness
        ▼
Confidence Score
        │
        ▼
Sequence Pattern

In [3]:
# importing
from datetime import datetime, timedelta
from collections import defaultdict, Counter
from itertools import combinations

import json
import math
import statistics
import random

random.seed(42)

# Dataset

Instead of using a small toy example,
this notebook uses a synthetic smart-home event log spanning
30 consecutive days.

The dataset intentionally contains

- recurring routines,
- missing actions,
- reordered actions,
- duplicate sensor updates,
- random unrelated events.

This allows the robustness of the algorithm to be evaluated under
realistic operating conditions.

# Configuration

For this notebook we simulate a rolling analysis window of fifteen days.

The values below define the core parameters used throughout the notebook.

In [6]:
ANALYSIS_WINDOW_DAYS = 15

MAX_SESSION_GAP_MINUTES = 10

MIN_SEQUENCE_LENGTH = 2

MAX_SEQUENCE_LENGTH = 8

HOUSEHOLD_ID = "H001"

# Loading the Dataset

Instead of generating random events, we use a carefully designed synthetic event log representing fifteen days of household activity.

The dataset contains recurring behavioural routines including

- Morning Departure
- Water Motor Operation
- Evening Arrival
- Night Routine
- Porch Lighting
- Kitchen Activity

At this stage, the dataset is intentionally kept clean so that the correctness of the mining algorithm can be validated before introducing behavioural imperfections.

## Inspecting the Dataset

Before applying any mining algorithm, it is important to understand the characteristics of the event log.

We begin by examining

- date range
- device types
- rooms
- actions
- household information

This step is equivalent to an exploratory data analysis (EDA) phase and helps verify that the data has been loaded correctly.

In [8]:
with open("events_15_days.json", "r") as f:
    events = json.load(f)

print(f"Total Events : {len(events)}")

Total Events : 281


In [9]:
timestamps = [
    datetime.fromisoformat(event["timestamp"])
    for event in events
]

print("Date Range")
print("-" * 40)
print("Start :", min(timestamps))
print("End   :", max(timestamps))

print()

print("Unique Devices")
print("-" * 40)

devices = sorted({
    event["device_id"]
    for event in events
})

for device in devices:
    print(device)

print()

print("Rooms")
print("-" * 40)

rooms = sorted({
    event["room"]
    for event in events
})

print(rooms)

Date Range
----------------------------------------
Start : 2026-06-01 06:00:00
End   : 2026-06-15 22:37:00

Unique Devices
----------------------------------------
bedroom_ac
bedroom_fan
bedroom_light
bedroom_presence
kitchen_light
livingroom_light
livingroom_tv
main_door
porch_light
water_motor

Rooms
----------------------------------------
['bedroom', 'entrance', 'kitchen', 'living_room', 'porch', 'utility']


In [10]:
device_counts = Counter(
    event["device_id"]
    for event in events
)

print(device_counts)

Counter({'bedroom_presence': 45, 'porch_light': 30, 'bedroom_fan': 30, 'bedroom_light': 30, 'main_door': 30, 'water_motor': 30, 'livingroom_tv': 30, 'kitchen_light': 26, 'livingroom_light': 15, 'bedroom_ac': 15})


### Why perform this analysis?

Although this information is not directly used during sequence mining, it serves several important purposes.

1. It verifies that the dataset has been loaded correctly.

2. It provides an overview of the smart-home environment.

3. It helps identify heavily used devices that may participate in multiple routines.

4. It establishes the baseline characteristics of the dataset before any processing begins.

In [11]:
events = sorted(
    events,
    key=lambda event: datetime.fromisoformat(event["timestamp"])
)

print("First Event")
print(events[0])

print()

print("Last Event")
print(events[-1])

First Event
{'event_id': 'E0001', 'household_id': 'H001', 'device_id': 'porch_light', 'device_type': 'light', 'room': 'porch', 'action': 'OFF', 'triggered_by': 'system', 'timestamp': '2026-06-01T06:00:00'}

Last Event
{'event_id': 'E0281', 'household_id': 'H001', 'device_id': 'bedroom_presence', 'device_type': 'presence', 'room': 'bedroom', 'action': 'ENTER', 'triggered_by': 'son', 'timestamp': '2026-06-15T22:37:00'}


# End of Part 1

At this stage the event log has been

- loaded,
- validated,
- explored,
- and chronologically sorted.

The resulting ordered event stream forms the input to the next stage of the pipeline.

In Part 2 we will construct **sessions**, where temporally related events are grouped into meaningful behavioural episodes before sequence mining begins.

# Part 2 — Session Construction

Individual IoT events rarely represent complete household behaviours.

Instead, routines occur as bursts of related actions happening within a short
period of time.

For example,

08:00  Bedroom Presence → LEAVE

08:02  Bedroom Fan → OFF

08:04  Bedroom Light → OFF

08:06  Main Door → LOCK

Although these are four independent events, together they represent one
behavioural routine.

The purpose of sessionization is to group temporally related events into
independent behavioural sessions that can later be compared for similarity.

## Session Definition

A session is defined as a chronologically ordered collection of events that
belong to the same household activity.

Two consecutive events belong to the same session if

1. the time gap from the previous event is sufficiently small, and

2. the overall duration of the session remains within a reasonable limit.

This prevents very long activities from becoming one giant session while still
allowing small pauses inside normal routines.

In [12]:
MAX_GAP_MINUTES = 10

MAX_SESSION_DURATION = 20

The sessionization algorithm processes the event stream sequentially.

For every event it determines whether the event should

• continue the current session, or

• begin a new session.

The decision is based on temporal proximity rather than event type, making the
algorithm independent of household configuration.

In [14]:
from datetime import datetime


def sessionize(events):

    sessions = []

    current_session = []

    session_start = None

    previous_time = None

    for event in events:

        current_time = datetime.fromisoformat(event["timestamp"])

        if not current_session:

            current_session = [event]
            session_start = current_time
            previous_time = current_time
            continue

        gap_previous = (
            current_time - previous_time
        ).total_seconds() / 60

        session_duration = (
            current_time - session_start
        ).total_seconds() / 60

        if (
            gap_previous <= MAX_GAP_MINUTES
            and # small pauses are allowed but sessions must be stopped from growing indefinetly. every session has a bdounded duration
            session_duration <= MAX_SESSION_DURATION
        ):

            current_session.append(event)

        else:

            sessions.append(current_session)

            current_session = [event]

            session_start = current_time

        previous_time = current_time

    if current_session:

        sessions.append(current_session)

    return sessions

In [15]:
sessions = sessionize(events)

print(f"Total Sessions : {len(sessions)}")

Total Sessions : 134


In [16]:
for i, session in enumerate(sessions[:5]):

    print("=" * 60)
    print(f"Session {i+1}")
    print("=" * 60)

    for event in session:

        print(
            event["timestamp"],
            " | ",
            event["device_id"],
            " | ",
            event["action"]
        )

    print()

Session 1
2026-06-01T06:00:00  |  porch_light  |  OFF

Session 2
2026-06-01T08:05:00  |  bedroom_presence  |  LEAVE
2026-06-01T08:07:00  |  bedroom_fan  |  OFF
2026-06-01T08:09:00  |  bedroom_light  |  OFF
2026-06-01T08:11:00  |  main_door  |  LOCK

Session 3
2026-06-01T09:00:00  |  water_motor  |  ON

Session 4
2026-06-01T09:15:00  |  water_motor  |  OFF

Session 5
2026-06-01T13:17:00  |  kitchen_light  |  ON



## Observations

The sessionization process transforms a continuous event stream into a collection
of independent behavioural sessions.

Each session now represents one candidate routine.

These sessions will form the fundamental unit of comparison throughout the
remainder of the sequence mining pipeline.

In the next stage, redundant sensor updates will be removed and each session
will be converted into a compact canonical representation suitable for
similarity computation.